### 1. Input

In [15]:
# ! pip install SpeechRecognition pydub
# ! pip install langdetect
# ! pip install openai-whisper
# ! pip install torch
import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect

#### 1.1. S2T

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Modelo base
model = w.load_model("base", device=device)

def input_speech(audio = None):
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente)
        result = model.transcribe(audio)
        # Se devuelve la transcripción del audio
        return result["text"].strip(), result["language"]
    
    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio y eliminar esa parte
            r.adjust_for_ambient_noise(source, duration=1)
            # Se escucha el audio
            audio_data = r.listen(source)
            
            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())
                
            result = model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma
            return result["text"].strip(), result["language"]

#### 1.2. T2T

In [ ]:
def input_text():
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma
    try:
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"
    
    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

#### 1.3. Cleaning text